# AI Coloring Book — Final Prototype

This notebook runs the complete **name → grounded biography → FLUX line art → PDF book** pipeline. Select a Colab **L4 GPU** and preferably a High-RAM runtime before starting. Stage outputs are cached, so rerunning the final cell resumes incomplete work.


In [ ]:
# Confirm that Colab assigned a GPU. An L4 is the tested target.
!nvidia-smi

In [ ]:
from pathlib import Path

REPOSITORY = 'https://github.com/icynic/AI-coloring-book.git'
PROJECT_DIR = Path('/content/AI-coloring-book')
if not PROJECT_DIR.exists():
    !git clone -q {REPOSITORY} {PROJECT_DIR}
%cd /content/AI-coloring-book
!git rev-parse --short HEAD

In [ ]:
# Install the environment pinned for the final prototype.
!pip install -q -r requirements-colab.txt

## Configure the run
The models are loaded sequentially, not simultaneously. Google Drive is recommended because Colab runtimes can disconnect. Set `FORCE_REGENERATE=True` only when you deliberately want to overwrite cached stage outputs.

In [ ]:
NAMES = [
    'Marie Curie',
    'Albert Einstein',
]

USE_GOOGLE_DRIVE = True
FORCE_REGENERATE = False
SEED = 42
QWEN_QUANTIZATION = 'none'   # use '4bit' if memory is tight
FLUX_QUANTIZATION = 'none'   # use '8bit' if memory is tight
FLUX_OFFLOAD = False         # enable if the unquantized model is close to OOM

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/AIColoringBook/final_run'
else:
    OUTPUT_DIR = '/content/AIColoringBook/final_run'

print('Output directory:', OUTPUT_DIR)

In [ ]:
# Run or resume the complete pipeline.
from main import main as run_coloring_book

arguments = [
    '--names', *NAMES,
    '--output-dir', OUTPUT_DIR,
    '--seed', str(SEED),
    '--qwen-quantization', QWEN_QUANTIZATION,
    '--flux-quantization', FLUX_QUANTIZATION,
]
if FLUX_OFFLOAD:
    arguments.append('--flux-offload')
if FORCE_REGENERATE:
    arguments.append('--force')

run_coloring_book(arguments)

In [ ]:
# Inspect the reproducibility record and display the finished book link.
import json
from IPython.display import FileLink, display

manifest_path = Path(OUTPUT_DIR) / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps(manifest['runtime'], indent=2))
for item in manifest['items']:
    print(item['title'], 'OK' if not item['errors'] else item['errors'])

book_path = Path(OUTPUT_DIR) / 'coloring_book.pdf'
display(FileLink(str(book_path)))